# 00 · Preparar el proyecto en Google Colab
Este notebook conecta código (GitHub), datos y resultados persistentes (Drive) con el cómputo temporal de Colab. No entrena un modelo.

Necesita una cuenta Google con acceso a la carpeta compartida y acceso al repositorio. Abra **Entorno de ejecución → Conectar** y ejecute cada celda, de arriba hacia abajo, con ▶. CPU es suficiente para esta preparación.

Complete únicamente la sección 1. Al terminar tendrá `PATHS`, `DATA_PATH`, `MANIFEST`, `EXPERIMENT_CONFIG` y `REPO_STATE`. Las variables pertenecen a esta sesión: otros notebooks deben cargar `session.json` dentro de la misma máquina o repetir este setup en una sesión nueva. Ante un error, deténgase y corrija la causa; no continúe hasta ver la comprobación esperada.

Guía del repositorio: `docs/guia_equipo.md`. Los resultados importantes se guardan explícitamente en Drive; el disco local puede desaparecer.


## 1 · Configuración única
Pegue la URL HTTPS real de GitHub, indique la rama y la ruta real de Drive. La carpeta compartida debe existir; puede añadir un acceso directo en **Mi unidad**. `dataset_relative_path` se resuelve dentro de `raw_dir`. Los nombres raw/processed/results/weights son una propuesta configurable, no carpetas que se presupongan existentes.

Indique una versión de datos estable. Para un ZIP active copia y extracción; para carpetas puede leer directamente desde Drive. Actualizar código es opcional y requiere una copia sin cambios locales.

Repositorio privado: cree en el panel **Secretos** de Colab un secreto `GITHUB_TOKEN` con un token de lectura limitado a este repositorio y autorice su acceso al notebook. Active `private_repo`. No pegue tokens en estas celdas, URLs ni salidas. El token solo pasa al proceso de Git en memoria; el helper temporal no contiene el token.


In [ ]:
SETTINGS = {
    "repo_url": "",  # Obligatorio: URL HTTPS del repositorio GitHub, sin credenciales.
    "branch": "",  # Obligatorio: nombre real de la rama.
    "drive_root": "",  # Obligatorio: ruta bajo /content/drive (use el explorador).
    "dataset_relative_path": "",  # Obligatorio: ruta dentro de raw_dir.
    "dataset_version": "",  # Obligatorio: identificador acordado por el equipo.
    "work_dir": "/content/cv-work",
    "raw_dir": "raw", "processed_dir": "processed",
    "results_dir": "results", "weights_dir": "weights",
    "seed": 42,  # El futuro entrenamiento deberá aplicar esta semilla.
    "hyperparameters": {},  # Completar cuando exista un modelo.
    "dataset_description": "",  # Procedencia real/simulada y notas.
    "splits": {},  # Opcional: nombre -> archivo de lista relativo al dataset.
    "validator": None,  # Opcional: modulo:funcion cuando se defina el formato.
    "copy_to_local": False, "extract_zip": False,
    "update_code": False, "private_repo": False,
    "sample_size": 20,
}
print("Configuración cargada. Complete todos los campos obligatorios antes de continuar.")


## 2 · Comprobar el entorno
Ejecute esta celda para comprobar Python, paquetes y acelerador. Si no está en Colab, abra este archivo en https://colab.research.google.com mediante **Archivo → Abrir notebook → GitHub** o **Subir**. La falta de GPU no impide preparar datos. PyTorch es opcional y se conserva la versión proporcionada por Colab.


In [ ]:
import sys, platform, importlib.util, importlib.metadata, subprocess, json, os
from pathlib import Path
from urllib.parse import urlsplit
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if not IN_COLAB:
    raise RuntimeError("Este setup requiere Google Colab. Para uso local siga docs/guia_equipo.md.")
if sys.version_info < (3, 10):
    raise RuntimeError("Se requiere Python 3.10 o posterior; use un runtime actual de Colab.")
print("Python:", platform.python_version())
for package in ("torch", "torchvision", "Pillow"):
    try: print(package, importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError: print(package, "no instalado")
GPU = {"available": False, "device": "cpu"}
try:
    import torch
    GPU["available"] = torch.cuda.is_available()
    GPU["cuda_runtime"] = torch.version.cuda
    if GPU["available"]:
        GPU["device"] = torch.cuda.get_device_name(0)
except ImportError:
    print("PyTorch no está disponible; la infraestructura puede continuar en CPU.")
except Exception as exc:
    raise RuntimeError("PyTorch está instalado pero no funciona. Revise su compatibilidad antes de continuar.") from exc
print("Cómputo:", GPU)
for key in ("repo_url", "branch", "drive_root", "dataset_relative_path", "dataset_version"):
    if not SETTINGS[key].strip(): raise ValueError(f"Complete SETTINGS[{key!r}] en la sección 1.")
u = urlsplit(SETTINGS["repo_url"])
if u.scheme != "https" or u.hostname != "github.com" or u.username or u.password or u.query or u.fragment:
    raise ValueError("Use una URL HTTPS de github.com sin tokens, credenciales ni parámetros.")
WORK = Path(SETTINGS["work_dir"]).expanduser().resolve()
if not WORK.is_relative_to(Path("/content")) or WORK.is_relative_to(Path("/content/drive")) or WORK == Path("/content"):
    raise ValueError("work_dir debe ser un subdirectorio temporal de /content fuera de Drive.")
WORK.mkdir(parents=True, exist_ok=True)
REPO = WORK / "repo"
print("Entorno compatible; directorio temporal:", WORK)


## 3 · Clonar o comprobar el código
Se comprueban remoto y rama. Con `update_code=False` se conserva el commit local; con `True` solo se acepta una actualización sin conflictos ni commits divergentes. Los cambios locales se conservan. Si Git falla, revise acceso al repositorio y la guía de Git; no borre su trabajo.

Esta celda incluye el pequeño helper de Git porque todavía no se ha clonado el código del proyecto. Muestra rama, commit y cambios pendientes. Si acaba de actualizar código, vuelva a abrir el notebook desde esa rama para cargar también sus celdas nuevas.


In [ ]:
"""Operaciones Git conservadoras, también utilizables desde Colab."""
from pathlib import Path
import os
import subprocess


def git(repo, *args):
    result = subprocess.run(["git", "-C", str(repo), *args], text=True, capture_output=True,
                            env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if result.returncode:
        # Evitar imprimir remotos o diagnósticos que pudieran contener credenciales.
        raise RuntimeError("Git no pudo completar la operación. Revise acceso, rama y estado local; consulte docs/guia_equipo.md.")
    return result.stdout.strip()


def repository_state(repo):
    return {"commit": git(repo, "rev-parse", "HEAD"),
            "branch": git(repo, "branch", "--show-current"),
            "dirty": bool(git(repo, "status", "--porcelain"))}


def ensure_repository(url, branch, destination, update=False):
    destination = Path(destination).expanduser().resolve()
    if not branch or branch.startswith("-"):
        raise ValueError("Indique una rama válida.")
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        git(destination.parent, "clone", "--branch", branch, "--", url, str(destination))
    else:
        if not (destination / ".git").exists():
            raise ValueError("El directorio existe pero no es un repositorio. Elija otro.")
        if git(destination, "remote", "get-url", "origin").rstrip("/") != url.rstrip("/"):
            raise ValueError("La copia local corresponde a otro remoto. No se modificó.")
        state = repository_state(destination)
        if state["branch"] != branch:
            raise ValueError("La rama local difiere de la solicitada. Guarde sus cambios y cambie de rama manualmente.")
        if update:
            if state["dirty"]:
                raise ValueError("Hay cambios locales. Guárdelos mediante un commit antes de actualizar.")
            git(destination, "fetch", "origin", branch)
            git(destination, "merge", "--ff-only", "FETCH_HEAD")
    return repository_state(destination)

import tempfile
from contextlib import contextmanager

@contextmanager
def github_auth():
    if not SETTINGS["private_repo"]:
        yield
        return
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError("No se pudo acceder al secreto GITHUB_TOKEN. Revise el panel Secretos y su autorización.") from None
    keys = ("GIT_ASKPASS", "CV_GITHUB_TOKEN")
    previous = {key: os.environ.get(key) for key in keys}
    try:
        with tempfile.TemporaryDirectory() as folder:
            helper = Path(folder) / "askpass.py"
            helper.write_text('#!/usr/bin/env python3\nimport os, sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["CV_GITHUB_TOKEN"])\n')
            helper.chmod(0o700)
            os.environ["GIT_ASKPASS"] = str(helper)
            os.environ["CV_GITHUB_TOKEN"] = token
            yield
    finally:
        for key, value in previous.items():
            if value is None: os.environ.pop(key, None)
            else: os.environ[key] = value
        token = None

with github_auth():
    REPO_STATE = ensure_repository(SETTINGS["repo_url"], SETTINGS["branch"], REPO, SETTINGS["update_code"])
print("Repositorio:", SETTINGS["repo_url"], REPO_STATE)
if REPO_STATE["dirty"]: print("Atención: cambios locales conservados. Haga commit antes de registrar experimentos.")


## 4 · Instalar las dependencias generales
Se usa el Python de esta sesión y las dependencias del repositorio. Pip reutiliza las versiones instaladas que cumplen requisitos. No se instala PyTorch ni CUDA aquí. Si un futuro modelo los requiere, siga el selector oficial de https://pytorch.org/get-started/locally/ y documente la combinación probada.

Si la instalación falla, lea su salida y consulte la guía del equipo. No se reinicia la sesión automáticamente.


In [ ]:
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=REPO, check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError("Falló la instalación. Revise el mensaje de pip y docs/guia_equipo.md.") from exc
# Cargar las utilidades directamente desde src/, sin instalar un paquete editable.
sys.path.insert(0, str(REPO / "src")) if str(REPO / "src") not in sys.path else None
from cv_project.config import resolve_paths
from cv_project.validation import validate_dataset
from cv_project.preparation import prepare_dataset
from cv_project.experiments import start_run, save_artifact, finish_run
print("Importaciones correctas: cv_project y Pillow disponibles.")


## 5 · Autorizar Drive y resolver las rutas
Colab abrirá la autorización oficial de Google. Elija la cuenta a la que se compartió la carpeta. No introduzca contraseñas en celdas.

Si no encuentra la carpeta: abra Drive en el navegador, compruebe permisos y añada un acceso directo de la carpeta compartida en **Mi unidad**. Copie su ruta desde el explorador de Colab a la sección 1. Las unidades compartidas de una organización pueden aparecer en una ruta distinta; no se presuponen permisos idénticos para todos.

Un montaje fallido y una ruta inexistente producen mensajes diferentes. Algunos fallos de permisos se presentan como rutas inexistentes: en ese caso compruebe también el acceso en el navegador.


In [ ]:
from google.colab import drive
try:
    drive.mount("/content/drive", force_remount=False)
except Exception:
    raise RuntimeError("Falló el montaje/autorización de Drive. Revise la cuenta y reintente la autorización oficial.") from None
MANIFEST = {key: SETTINGS[key] for key in ("raw_dir", "processed_dir", "results_dir", "weights_dir")}
MANIFEST["dataset"] = {
    "relative_path": SETTINGS["dataset_relative_path"],
    "version": SETTINGS["dataset_version"],
    "description": SETTINGS["dataset_description"],
    "sample_size": SETTINGS["sample_size"],
    "splits": SETTINGS["splits"], "validator": SETTINGS["validator"],
}
try:
    PATHS = resolve_paths(SETTINGS["drive_root"], MANIFEST)
    print("Contenido de la carpeta compartida:", sorted(p.name for p in PATHS["root"].iterdir())[:50])
except PermissionError:
    raise PermissionError("Drive está montado, pero la cuenta no puede leer esta carpeta. Solicite acceso al propietario.") from None
EXPERIMENT_CONFIG = {"seed": SETTINGS["seed"], "hyperparameters": SETTINGS["hyperparameters"]}
print("Rutas resueltas:", {k: str(v) for k, v in PATHS.items()})


## 6 · Inspeccionar el dataset original
La validación es de solo lectura: cuenta archivos, extensiones, bytes y revisa una muestra determinista de imágenes. No interpreta etiquetas. Una validación aprobada no garantiza calidad científica ni integridad de todas las imágenes. Para formatos concretos agregue una función `modulo:funcion` en `validator` de la sección 1, según la guía del equipo.

Se espera `ok: true`. Un ZIP se inspeccionará como archivo; su contenido se validará tras la extracción opcional.


In [ ]:
SOURCE_REPORT = validate_dataset(PATHS["dataset"], MANIFEST["dataset"]["sample_size"])
print(json.dumps(SOURCE_REPORT, ensure_ascii=False, indent=2))
if not SOURCE_REPORT["ok"]:
    raise ValueError("El dataset original requiere correcciones. Revise errors; no se modificó ningún archivo.")


## 7 · Preparación opcional en disco local
Para muchas lecturas, use `copy_to_local=True`. La utilidad comprueba espacio y hashes, reutiliza una copia idéntica y rechaza cambios en el origen o destino; elija un `work_dir` nuevo si cambia de dataset. Calcular hashes puede tardar en datasets grandes.

La extracción solo admite ZIP y debe solicitarse con `extract_zip=True`. Rechaza rutas inseguras y no extrae sobre un directorio existente. No borra originales. Sin copia se trabaja directamente sobre Drive; no escriba desde su entrenamiento sobre esos datos.

La copia se guarda dentro del clon en `data/prepared/data/`, excluida de Git. `DATA_PATH` apunta a esa copia o al dataset de Drive si no activa la copia. El código de carga debe usar `DATA_PATH` en ambos casos.


In [ ]:
if SETTINGS["extract_zip"] and not SETTINGS["copy_to_local"]:
    raise ValueError("Para extraer un ZIP active también copy_to_local.")
DATA_PATH = PATHS["dataset"]
if SETTINGS["copy_to_local"]:
    DATA_PATH = prepare_dataset(DATA_PATH, REPO / "data" / "prepared", extract_zip=SETTINGS["extract_zip"])
REPORT = validate_dataset(DATA_PATH, MANIFEST["dataset"]["sample_size"], MANIFEST["dataset"].get("validator"))
print(json.dumps(REPORT, ensure_ascii=False, indent=2))
if not REPORT["ok"]: raise ValueError("La validación final falló; revise errors antes de entrenar.")
print("Datos preparados:", DATA_PATH, "· bytes:", REPORT["bytes"])


## 8 · Salidas persistentes y resumen
Esta celda crea únicamente las carpetas de resultados y pesos y comprueba escritura. Guarda `session.json` en el directorio temporal para reutilizar rutas desde código de esta misma sesión. Repita el setup cuando Colab asigne otra máquina.

El estado «preparación completada» confirma la infraestructura, no la disponibilidad de un modelo ni la validez de etiquetas. Si conserva un ZIP sin extraer, su futuro cargador deberá admitirlo.


In [ ]:
for key in ("results", "weights"):
    PATHS[key].mkdir(parents=True, exist_ok=True)
    try:
        with tempfile.TemporaryFile(dir=PATHS[key]) as probe:
            probe.write(b"write-check")
            probe.flush()
    except OSError as exc:
        raise OSError(f"No se pudo escribir en {PATHS[key]}; revise permisos y cuota de Drive.") from exc
SUMMARY = {"repo": SETTINGS["repo_url"], "repo_dir": str(REPO), "git": REPO_STATE, "python": platform.python_version(),
           "gpu": GPU, "data_path": str(DATA_PATH), "validation_ok": REPORT["ok"],
           "paths": {k: str(v) for k, v in PATHS.items()}, "manifest": MANIFEST,
           "experiment_config": EXPERIMENT_CONFIG, "status": "Preparación completada; modelo pendiente"}
(WORK / "session.json").write_text(json.dumps(SUMMARY, ensure_ascii=False, indent=2))
print(json.dumps(SUMMARY, ensure_ascii=False, indent=2))
print("Estado temporal guardado en:", WORK / "session.json")


## 9 · Guardar resultados explícitamente cuando exista un experimento
La celda siguiente solo muestra el patrón de uso; no inventa métricas ni crea una ejecución de entrenamiento. Antes de un experimento real haga commit de su código, complete hiperparámetros, versión de datos y particiones. `start_run` rechaza cambios sin commit.

Llame a `start_run` al iniciar; durante el entrenamiento exporte checkpoints seleccionados con nombres distintos. Al finalizar copie pesos y visualizaciones deseadas con `save_artifact`, y cierre el registro con `finish_run`. Las colisiones se rechazan con `FileExistsError`; no vuelva a finalizar el mismo registro. Cada ejecución nueva tiene un identificador único. Compruebe archivos en el navegador de Drive antes de desconectar.

Los metadatos y versiones del entorno quedan en resultados; los pesos en su carpeta separada. No se sincroniza automáticamente el disco temporal.


In [ ]:
print("Utilidades disponibles: start_run, save_artifact, finish_run.")
# Cuando exista un experimento real, ejecute estas llamadas en su código:
# RUN = start_run(PATHS, REPO, MANIFEST, EXPERIMENT_CONFIG, DATA_PATH)
# artifacts = [save_artifact(ruta_peso_final, PATHS["weights"] / RUN.name)]
# artifacts.append(save_artifact(ruta_visualizacion, RUN))
# record = finish_run(RUN, metrics=metricas_reales, artifacts=artifacts)
# print(record)


## 10 · Continuar con los notebooks por pasos
Los notebooks 01 y 02 se mantienen separados. Para ejecutarlos en esta misma máquina de Colab, use la función de abajo en una nueva celda. Así no pierde el entorno ni tiene que repetir la configuración. Puede abrir cada notebook para leer sus explicaciones; ejecutarlo en otra máquina requeriría preparar esa sesión de nuevo.

Después de ejecutar esta celda, añada otra con `run_step("01_exploracion_datos.ipynb")`. Cuando existan resultados, use `run_step("02_evaluacion.ipynb")`. La evaluación actual solo consulta registros; aún no hay modelo entrenado.


In [ ]:
def run_step(name):
    if name not in {"01_exploracion_datos.ipynb", "02_evaluacion.ipynb"}:
        raise ValueError("Elija el notebook 01 de exploración o el 02 de evaluación.")
    from IPython.display import display, Markdown
    notebook = json.loads((REPO / "notebooks" / name).read_text())
    namespace = {"SESSION_FILE": WORK / "session.json"}
    for cell in notebook["cells"]:
        source = cell["source"]
        if isinstance(source, list): source = "".join(source)
        if cell["cell_type"] == "markdown": display(Markdown(source))
        elif cell["cell_type"] == "code": exec(compile(source, name, "exec"), namespace)
    return namespace

print('Siguiente paso: ejecute run_step("01_exploracion_datos.ipynb") en una nueva celda.')
